In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')
df_raw = df.copy()   # untouched copy for the before/after table
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
print("Shape:", df.shape)
df.info()

Shape: (10000, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [3]:
placeholders = ['ERROR', 'UNKNOWN']

report = pd.DataFrame({
    'Nulls': df.isnull().sum(),
    'ERROR/UNKNOWN': df.isin(placeholders).sum()
})
report['Total missing'] = report['Nulls'] + report['ERROR/UNKNOWN']
report['% missing'] = (report['Total missing'] / len(df) * 100).round(1)
report

,Nulls,ERROR/UNKNOWN,Total missing,% missing
Transaction ID,0,0,0,0.0
Item,333,636,969,9.7
Quantity,138,341,479,4.8
Price Per Unit,179,354,533,5.3
Total Spent,173,329,502,5.0
Payment Method,2579,599,3178,31.8
Location,3265,696,3961,39.6
Transaction Date,159,301,460,4.6


In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Transaction IDs:", df['Transaction ID'].duplicated().sum())


Duplicate rows: 0
Duplicate Transaction IDs: 0


## Data Quality Report: Findings

- Every column except Transaction ID has missing data, and a lot of it is hidden. Besides real nulls, the dataset uses the text placeholders ERROR and UNKNOWN, which isnull() does not catch. Counting both, Location is missing about 40% of values, Payment Method about 32%, Item about 10%, and the numeric columns and date about 5% each.
- Quantity, Price Per Unit, Total Spent and Transaction Date are stored as text (because of the placeholders), so they have the wrong dtype.
- There are no duplicate rows and no duplicate Transaction IDs.

In [8]:
# strip stray whitespace from text columns
for col in ['Item', 'Payment Method', 'Location']:
    df[col] = df[col].str.strip()

# turn ERROR / UNKNOWN into proper missing values
df = df.replace(placeholders, np.nan)

# fix data formats
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

df.isnull().sum()


Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [9]:
df[['Quantity', 'Price Per Unit', 'Total Spent']].describe()

,Quantity,Price Per Unit,Total Spent
count,9521.000000,9467.000000,9498.000000
mean,3.028463,2.949984,8.924352
std,1.419007,1.278450,6.009919
min,1.000000,1.000000,1.000000
25%,2.000000,2.000000,4.000000
50%,3.000000,3.000000,8.000000
75%,4.000000,4.000000,12.000000
max,5.000000,5.000000,25.000000


## Missing Data Strategy

- Price Per Unit: every item has one fixed menu price (e.g. Coffee 2.0, Cake 3.0, Tea 1.5), so missing prices are recovered from the Item name instead of guessed.
- Quantity: on complete rows, Total Spent always equals Quantity x Price Per Unit, so a missing Quantity is recovered as Total Spent / Price Per Unit.
- Total Spent: recovered as Quantity x Price Per Unit.
- Rows where two of the three numeric values are missing cannot be recovered, so those rows are dropped (a small share of the data).
- Transaction Date: cannot be inferred from any other column. It is missing in under 5% of rows, so I drop those rows rather than invent dates.
- Item, Payment Method, Location: these are categories, and Payment Method and Location are missing in about a third or more of rows, so dropping them would throw away too much good data. I fill them with an explicit "Unknown" category instead, which keeps the rows and keeps the gap visible.

In [11]:
# Price: recover from Item (each item has one price)
price_map = (df.dropna(subset=['Item', 'Price Per Unit'])
               .groupby('Item')['Price Per Unit']
               .agg(lambda x: x.mode()[0]))
print(price_map)

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].map(price_map))

Item
Cake        3.0
Coffee      2.0
Cookie      1.0
Juice       3.0
Salad       5.0
Sandwich    4.0
Smoothie    4.0
Tea         1.5
Name: Price Per Unit, dtype: float64


In [12]:
# Quantity: recover from Total Spent / Price Per Unit
mask = df['Quantity'].isna() & df['Total Spent'].notna() & df['Price Per Unit'].notna()
print("Quantities recovered:", mask.sum())
df.loc[mask, 'Quantity'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Price Per Unit']

# Total Spent: recover from Quantity x Price
df['Total Spent'] = df['Total Spent'].fillna(df['Quantity'] * df['Price Per Unit'])

df[['Quantity', 'Price Per Unit', 'Total Spent']].isnull().sum()

Quantities recovered: 456


Quantity          23
Price Per Unit    54
Total Spent       23
dtype: int64

In [13]:
# rows that still can't be recovered: drop
before = len(df)
df = df.dropna(subset=['Quantity', 'Price Per Unit', 'Total Spent'])
print("Dropped (unrecoverable numeric data):", before - len(df))

before = len(df)
df = df.dropna(subset=['Transaction Date'])
print("Dropped (missing date):", before - len(df))

# categories: explicit Unknown label
for col in ['Item', 'Payment Method', 'Location']:
    df[col] = df[col].fillna('Unknown')

df.isnull().sum()

Dropped (unrecoverable numeric data): 74
Dropped (missing date): 457


Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [14]:
before = len(df)
df = df.drop_duplicates()
df = df.drop_duplicates(subset='Transaction ID')
print("Duplicates removed:", before - len(df))

Duplicates removed: 0


In [15]:
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    print(f"{col}: bounds [{low}, {high}], outliers = {n}")

Quantity: bounds [-1.0, 7.0], outliers = 0
Price Per Unit: bounds [-1.0, 7.0], outliers = 0
Total Spent: bounds [-8.0, 24.0], outliers = 259


In [16]:
# what are the Total Spent "outliers"?
df[df['Total Spent'] > 24]['Total Spent'].value_counts()

25.0    259
Name: Total Spent, dtype: int64

## Outlier Decision

IQR flags no outliers in Quantity or Price Per Unit. It flags 259 rows in Total Spent, but all of them equal exactly 25, which is the highest possible total (5 items x the most expensive 5.0 menu price). These are legitimate large orders, not errors, so I retain all of them.

In [19]:
df['Quantity'] = df['Quantity'].astype(int)
df['Transaction ID'] = df['Transaction ID'].astype(str)
for col in ['Item', 'Payment Method', 'Location']:
    df[col] = df[col].astype('category')

df.dtypes

Transaction ID              object
Item                      category
Quantity                     int32
Price Per Unit             float64
Total Spent                float64
Payment Method            category
Location                  category
Transaction Date    datetime64[ns]
dtype: object

In [20]:
def dtype_score(d):
    checks = [
        pd.api.types.is_numeric_dtype(d['Quantity']),
        pd.api.types.is_numeric_dtype(d['Price Per Unit']),
        pd.api.types.is_numeric_dtype(d['Total Spent']),
        pd.api.types.is_datetime64_any_dtype(d['Transaction Date'])
    ]
    return f"{sum(checks)}/4"

summary = pd.DataFrame({
    'Metric': ['Rows', 'Null values', 'ERROR/UNKNOWN placeholders',
               'Duplicate rows', 'Columns with correct dtype'],
    'Before': [len(df_raw), int(df_raw.isnull().sum().sum()),
               int(df_raw.isin(placeholders).sum().sum()),
               int(df_raw.duplicated().sum()), dtype_score(df_raw)],
    'After':  [len(df), int(df.isnull().sum().sum()),
               int(df.isin(placeholders).sum().sum()),
               int(df.duplicated().sum()), dtype_score(df)]
})
summary

,Metric,Before,After
0,Rows,10000,9469
1,Null values,6826,0
2,ERROR/UNKNOWN placeholders,3256,0
3,Duplicate rows,0,0
4,Columns with correct dtype,0/4,4/4


In [21]:
df.to_csv('cleaned_cafe_sales.csv', index=False)
pd.read_csv('cleaned_cafe_sales.csv').shape

(9469, 8)